# Physics-Informed Digital Twin for Turbojet Health Monitoring (HAL PS-2)

This notebook contains the complete implementation of the **Physics-Informed Neural Network (PINN)** Digital Twin designed to reconstruct the diagnostic and performance states of a single-spool four-stage turbojet engine.

### Core Methodology:
1. **Calorically Semi-Perfect Gas Model**: Locally calculates temperature-dependent specific heats $c_p(T)$ to yield realistic isentropic compressor and turbine operations.
2. **Differentiable Physics Regularization**: Implements physical constraints directly inside the training loop to guide predictions on unseen operational telemetry.
3. **Uncertainty Quantification**: Calculates prediction margins of error using validation Root Mean Squared Error (RMSE) values.
4. **Parameter Export**: Generates the JavaScript weights database (`weights.js`) to power the client-side diagnostic dashboard.

In [1]:
# 1. Import libraries
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [2]:
# 2. Load the telemetry and ground truth datasets
data_dir = "datasets"
train_feat = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_feat = pd.read_csv(os.path.join(data_dir, 'test.csv'))
gt = pd.read_csv(os.path.join(data_dir, 'ground_truth.csv'))

# Merge inputs with target health states
train_df = pd.merge(train_feat, gt, on=['EngineID', 'Cycle'], how='inner')
test_df = pd.merge(test_feat, gt, on=['EngineID', 'Cycle'], how='inner')

print(f"Loaded train shape: {train_df.shape}")
print(f"Loaded test shape: {test_df.shape}")

Loaded train shape: (240, 20)
Loaded test shape: (60, 20)


In [3]:
# 3. Preprocess and Scale Features
feature_cols = [
    'Cycle', 'Altitude_m', 'Mach', 'Tamb_K', 'Pamb_Pa', 'RPM_rev_min', 
    'FuelFlow_kg_s', 'P2_Pa', 'T2_K', 'P3_Pa', 'T3_K', 'P4_Pa', 'T4_K'
]
target_cols = [
    'CompressorHealth', 'CombustorHealth', 'TurbineHealth', 
    'OverallHealth', 'Thrust_N', 'TSFC_g_N_s'
]

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(train_df[feature_cols])
X_test_scaled = scaler_X.transform(test_df[feature_cols])

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(train_df[target_cols])
y_test_scaled = scaler_y.transform(test_df[target_cols])

# Scalers properties to PyTorch tensors
y_mean = torch.tensor(scaler_y.mean_, dtype=torch.float32)
y_scale = torch.tensor(scaler_y.scale_, dtype=torch.float32)
X_mean = torch.tensor(scaler_X.mean_, dtype=torch.float32)
X_scale = torch.tensor(scaler_X.scale_, dtype=torch.float32)

# Training tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test_scaled, dtype=torch.float32)

In [4]:
# 4. Define neural network backbone
class TurbojetPINN(nn.Module):
    def __init__(self, input_dim=13, output_dim=6):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, output_dim)
        )
        
    def forward(self, x):
        return self.network(x)

In [5]:
# 5. Define Differentiable Physics-Informed Custom Loss Function
class BraytonPINNLoss(nn.Module):
    def __init__(self, y_mean, y_scale, X_mean, X_scale, feature_cols):
        super().__init__()
        self.register_buffer('y_mean', y_mean)
        self.register_buffer('y_scale', y_scale)
        self.register_buffer('X_mean', X_mean)
        self.register_buffer('X_scale', X_scale)
        
        self.fuel_flow_idx = feature_cols.index('FuelFlow_kg_s')
        self.Tamb_idx = feature_cols.index('Tamb_K')
        self.Pamb_idx = feature_cols.index('Pamb_Pa')
        self.T2_idx = feature_cols.index('T2_K')
        self.P2_idx = feature_cols.index('P2_Pa')
        self.T3_idx = feature_cols.index('T3_K')
        self.P3_idx = feature_cols.index('P3_Pa')
        self.T4_idx = feature_cols.index('T4_K')
        self.P4_idx = feature_cols.index('P4_Pa')
        self.mse = nn.MSELoss()

    def forward(self, pred_scaled, target_scaled, x_scaled, 
                lambda_health=10.0, lambda_tsfc=100.0, lambda_brayton=1.0):
        loss_data = self.mse(pred_scaled, target_scaled)
        
        # Unscale to physical units
        pred_unscaled = pred_scaled * self.y_scale + self.y_mean
        comp_h, comb_h, turb_h, over_h, thrust, tsfc = [pred_unscaled[:, i] for i in range(6)]
        
        x_unscaled = x_scaled * self.X_scale + self.X_mean
        fuel_flow = x_unscaled[:, self.fuel_flow_idx]
        Tamb = x_unscaled[:, self.Tamb_idx]
        Pamb = x_unscaled[:, self.Pamb_idx]
        T2 = x_unscaled[:, self.T2_idx]
        P2 = x_unscaled[:, self.P2_idx]
        T3 = x_unscaled[:, self.T3_idx]
        P3 = x_unscaled[:, self.P3_idx]
        T4 = x_unscaled[:, self.T4_idx]
        P4 = x_unscaled[:, self.P4_idx]
        
        # Empirical losses
        loss_physics_health = torch.mean((over_h - (0.4*comp_h + 0.3*comb_h + 0.3*turb_h)) ** 2)
        loss_physics_tsfc = torch.mean((tsfc - (fuel_flow * 1000.0) / (thrust + 1e-8)) ** 2)
        
        # Specific heat calculation (J/(kg*K))
        def get_k(T):
            cp = 1005.0 + 0.05 * T + 1.5e-4 * (T ** 2) - 8.0e-8 * (T ** 3)
            return 287.058 / cp

        # Compressor isentropic efficiency loss
        T_comp_avg = (Tamb + T2) / 2.0
        kc = get_k(T_comp_avg)
        T2_isen = Tamb * torch.pow(torch.clamp(P2 / (Pamb + 1e-8), min=0.1), kc)
        expected_comp_h = torch.clamp((T2_isen - Tamb) / (T2 - Tamb + 1e-8), 0.5, 1.0)
        loss_brayton_comp = torch.mean((comp_h - expected_comp_h) ** 2)
        
        # Turbine isentropic efficiency loss
        T_turb_avg = (T3 + T4) / 2.0
        kt = get_k(T_turb_avg)
        T4_isen = T3 * torch.pow(torch.clamp(P4 / (P3 + 1e-8), min=0.01, max=1.0), kt)
        expected_turb_h = torch.clamp((T3 - T4) / (T3 - T4_isen + 1e-8), 0.3, 1.0)
        loss_brayton_turb = torch.mean((turb_h - expected_turb_h) ** 2)

        # Combustor pressure drop relative loss (Target ratio: 0.949)
        loss_brayton_comb_p = torch.mean(((P3 - 0.949 * P2) / (P2 + 1e-8)) ** 2)
        
        total_brayton_loss = loss_brayton_comp + loss_brayton_comb_p + loss_brayton_turb
        total_loss = loss_data + lambda_health*loss_physics_health + lambda_tsfc*loss_physics_tsfc + lambda_brayton*total_brayton_loss
        return total_loss

In [6]:
# 6. Instantiate & train the model
model = TurbojetPINN()
criterion = BraytonPINNLoss(y_mean, y_scale, X_mean, X_scale, feature_cols)
optimizer = optim.Adam(model.parameters(), lr=0.005)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=800, eta_min=5e-4)

print("Beginning PINN training loop...")
epochs = 800
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t, X_train_t)
    
    loss.backward()
    optimizer.step()
    scheduler.step()
    
    if epoch % 100 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} / {epochs:3d} | Loss: {loss.item():.5f}")

Beginning PINN training loop...
Epoch   1 / 800 | Loss: 1.11291


Epoch 100 / 800 | Loss: 0.16293


Epoch 200 / 800 | Loss: 0.14486


Epoch 300 / 800 | Loss: 0.13632


Epoch 400 / 800 | Loss: 0.12830


Epoch 500 / 800 | Loss: 0.12175
Epoch 600 / 800 | Loss: 0.11780


Epoch 700 / 800 | Loss: 0.11565


Epoch 800 / 800 | Loss: 0.11433


In [7]:
# 7. Model Evaluation
model.eval()
with torch.no_grad():
    preds_scaled = model(X_test_t).numpy()
    preds_unscaled = scaler_y.inverse_transform(preds_scaled)

y_test_unscaled = test_df[target_cols].values
print("="*60)
print("TEST PERFORMANCE METRIC MATRICES (CYCLE INCLUDED)")
print("="*60)
for i, col in enumerate(target_cols):
    rmse = np.sqrt(mean_squared_error(y_test_unscaled[:, i], preds_unscaled[:, i]))
    r2 = r2_score(y_test_unscaled[:, i], preds_unscaled[:, i])
    print(f"{col:<20} | RMSE: {rmse:>10.4f} | R2: {r2:>8.4f}")
print("="*60)

TEST PERFORMANCE METRIC MATRICES (CYCLE INCLUDED)
CompressorHealth     | RMSE:     0.0195 | R2:   0.9327
CombustorHealth      | RMSE:     0.0174 | R2:   0.6560
TurbineHealth        | RMSE:     0.0247 | R2:   0.7344
OverallHealth        | RMSE:     0.0113 | R2:   0.9516
Thrust_N             | RMSE:  1475.0036 | R2:   0.9913
TSFC_g_N_s           | RMSE:     0.0006 | R2:   0.9925


In [8]:
# 8. Save artifacts
os.makedirs('artifacts', exist_ok=True)
torch.save(model.state_dict(), 'artifacts/brayton_pinn_model.pt')
with open('artifacts/scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open('artifacts/scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)
print("All assets saved successfully!")

All assets saved successfully!


In [9]:
# 9. Inference with Uncertainty Bounds (Confidence Intervals)
# Let's use the takeoff condition preset for demonstration
takeoff_sample = [1, 0, 0, 288.15, 101325, 16300, 2.12, 610000, 595, 578000, 1850, 182000, 1310]

# Standard error bounds (95% CI is +/- 1.96 * RMSE)
rmse_bounds = {
    'CompressorHealth': 0.0195,
    'CombustorHealth': 0.0174,
    'TurbineHealth': 0.0247,
    'OverallHealth': 0.0113,
    'Thrust_N': 1476.7972,
    'TSFC_g_N_s': 0.0006
}

sample_scaled = scaler_X.transform([takeoff_sample])
with torch.no_grad():
    sample_pred_scaled = model(torch.tensor(sample_scaled, dtype=torch.float32)).numpy()
    sample_pred = scaler_y.inverse_transform(sample_pred_scaled)[0]

print("="*60)
print("PREDICTED STATES WITH 95% CONFIDENCE INTERVALS (UQ)")
print("="*60)
for i, col in enumerate(target_cols):
    val = sample_pred[i]
    bound = 1.96 * rmse_bounds[col]
    if 'Health' in col:
        print(f"{col:<20} | Prediction: {val*100:>5.1f}% | 95% CI: [{(val-bound)*100:>5.1f}%, {(val+bound)*100:>5.1f}%]")
    else:
        unit = 'N' if 'Thrust' in col else 'g/N/s'
        fmt = '.0f' if 'Thrust' in col else '.5f'
        print(f"{col:<20} | Prediction: {val:>{'10' if 'Thrust' in col else '7'}.{'0f' if 'Thrust' in col else '5f'}} {unit} | 95% CI: [{val-bound:>{'10' if 'Thrust' in col else '7'}.{'0f' if 'Thrust' in col else '5f'}}, {val+bound:>{'10' if 'Thrust' in col else '7'}.{'0f' if 'Thrust' in col else '5f'}}]")
print("="*60)

PREDICTED STATES WITH 95% CONFIDENCE INTERVALS (UQ)
CompressorHealth     | Prediction:  74.4% | 95% CI: [ 70.5%,  78.2%]
CombustorHealth      | Prediction:  85.4% | 95% CI: [ 82.0%,  88.8%]
TurbineHealth        | Prediction:  80.1% | 95% CI: [ 75.2%,  84.9%]
OverallHealth        | Prediction:  78.9% | 95% CI: [ 76.7%,  81.1%]
Thrust_N             | Prediction:      57293 N | 95% CI: [     54399,      60188]
TSFC_g_N_s           | Prediction: 0.02723 g/N/s | 95% CI: [0.02605, 0.02841]


C:\Users\Lenovo\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [10]:
# 10. Export parameters to weights.js for client-side evaluation
w1 = model.network[0].weight.detach().numpy().tolist()
b1 = model.network[0].bias.detach().numpy().tolist()
w2 = model.network[2].weight.detach().numpy().tolist()
b2 = model.network[2].bias.detach().numpy().tolist()
w3 = model.network[4].weight.detach().numpy().tolist()
b3 = model.network[4].bias.detach().numpy().tolist()
w4 = model.network[6].weight.detach().numpy().tolist()
b4 = model.network[6].bias.detach().numpy().tolist()

# Scale params
scaler_X_mean = scaler_X.mean_.tolist()
scaler_X_scale = scaler_X.scale_.tolist()
scaler_y_mean = scaler_y.mean_.tolist()
scaler_y_scale = scaler_y.scale_.tolist()

# Write to public/weights.js
weights_js_path = os.path.join('public', 'weights.js')
with open(weights_js_path, 'w') as f:
    f.write('// Dynamic model weights & scaling statistics\n')
    f.write(f'const scalerParams = {{\n')
    f.write(f'  scaler_X_mean: {scaler_X_mean},\n')
    f.write(f'  scaler_X_scale: {scaler_X_scale},\n')
    f.write(f'  scaler_y_mean: {scaler_y_mean},\n')
    f.write(f'  scaler_y_scale: {scaler_y_scale}\n')
    f.write(f'}};\n\n')
    f.write(f'const modelWeights = {{\n')
    f.write(f'  w1: {w1},\n')
    f.write(f'  b1: {b1},\n')
    f.write(f'  w2: {w2},\n')
    f.write(f'  b2: {b2},\n')
    f.write(f'  w3: {w3},\n')
    f.write(f'  b3: {b3},\n')
    f.write(f'  w4: {w4},\n')
    f.write(f'  b4: {b4}\n')
    f.write(f'}};\n')
print("Weights successfully exported to public/weights.js!")

Weights successfully exported to public/weights.js!
